# CM3070 FieldServe CRM — Scheduling Optimisation
**Goal:** Show that ML-predicted job durations + OR-Tools route optimisation meaningfully reduces travel time and increases daily job capacity vs unoptimised baselines.

**Dataset:** Olist Brazilian E-Commerce — reframed as field-service jobs  
**ML task:** Regression — predict job duration from order features  
**Optimisation:** Google OR-Tools VRP solver  
**Baselines:** (1) Random order, (2) Nearest-neighbour heuristic, (3) OR-Tools optimised  
**Metrics:** MAE on duration prediction; travel time reduction %, jobs completed per day

---
## Prerequisites
Same `data/olist/` folder used by `01_churn.ipynb`. You additionally need:
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_order_payments_dataset.csv`
- `olist_customers_dataset.csv`
- `olist_geolocation_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

DATA_DIR = Path('data/olist')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded ✓')

## 1  Load & Reframe Olist as Field-Service Jobs

| Olist concept | FieldServe concept |
|---|---|
| `order_id` | `job_id` |
| `order_purchase_timestamp` → `order_delivered_customer_date` | job start → job end |
| elapsed days (capped & scaled) | **job_duration_hours** (our regression target) |
| `customer_zip_code_prefix` → lat/lng | job site location |
| `price` + `freight_value` | job value proxy |

In [ ]:
orders    = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv',
                        parse_dates=['order_purchase_timestamp',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
items     = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments  = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
geo       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')

# ── aggregate items & payments to order level ──────────────────────────────
items_agg = items.groupby('order_id').agg(
    item_count   = ('order_item_id', 'count'),
    total_price  = ('price', 'sum'),
    total_freight= ('freight_value', 'sum')
).reset_index()

pay_agg = payments.groupby('order_id').agg(
    payment_value        = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type         = ('payment_type', 'first')
).reset_index()

# ── median lat/lng per zip prefix ─────────────────────────────────────────
geo_median = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')
).reset_index().rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix'})

# ── merge everything ───────────────────────────────────────────────────────
jobs = (
    orders
    .merge(customers[['customer_id','customer_unique_id',
                       'customer_zip_code_prefix','customer_state']], on='customer_id')
    .merge(items_agg,  on='order_id', how='left')
    .merge(pay_agg,    on='order_id', how='left')
    .merge(geo_median, on='customer_zip_code_prefix', how='left')
)

# keep only delivered orders with both timestamps
jobs = jobs[
    (jobs['order_status'] == 'delivered') &
    jobs['order_delivered_customer_date'].notna()
].copy()

# ── derive job_duration_hours (our regression target) ─────────────────────
# purchase → delivery elapsed time in hours, clipped to 1–72h range,
# then scaled to a realistic field-job window of 0.5–8 hours.
elapsed_h = (
    (jobs['order_delivered_customer_date'] - jobs['order_purchase_timestamp'])
    .dt.total_seconds() / 3600
)
elapsed_h = elapsed_h.clip(1, 720)  # remove outliers

# min-max scale to [0.5, 8.0] hours  (0.5h = quick clean; 8h = full-day job)
e_min, e_max = elapsed_h.min(), elapsed_h.max()
jobs['job_duration_hours'] = 0.5 + (elapsed_h - e_min) / (e_max - e_min) * 7.5

# ── calendar features ──────────────────────────────────────────────────────
jobs['hour_of_day']  = jobs['order_purchase_timestamp'].dt.hour
jobs['day_of_week']  = jobs['order_purchase_timestamp'].dt.dayofweek
jobs['month']        = jobs['order_purchase_timestamp'].dt.month
jobs['is_weekend']   = (jobs['day_of_week'] >= 5).astype(int)

# drop rows missing lat/lng (no location = can't route)
jobs = jobs.dropna(subset=['lat','lng','job_duration_hours'])

print(f'Jobs loaded: {len(jobs):,}')
print(f'Duration range: {jobs["job_duration_hours"].min():.2f}h – {jobs["job_duration_hours"].max():.2f}h')
print(f'Mean duration : {jobs["job_duration_hours"].mean():.2f}h')
jobs[['job_duration_hours','item_count','total_price','lat','lng']].describe().round(2)

## 2  Feature Engineering for Duration Prediction

In [ ]:
DURATION_FEATURES = [
    'item_count',
    'total_price',
    'total_freight',
    'payment_value',
    'payment_installments',
    'hour_of_day',
    'day_of_week',
    'month',
    'is_weekend',
]

TARGET = 'job_duration_hours'

df = jobs[DURATION_FEATURES + [TARGET]].dropna().copy()

X = df[DURATION_FEATURES].values
y = df[TARGET].values

print(f'Feature matrix: {X.shape}')

# duration distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(y, bins=50, color='#028090', edgecolor='white', alpha=0.85)
ax.set_xlabel('Job Duration (hours)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Job Durations')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'scheduling_duration_dist.png', dpi=150)
plt.show()

## 3  Train/Test Split & Models

Three regression models compared:
- **Baseline:** Mean predictor (predict the global mean for every job)
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

Primary metric: **MAE (Mean Absolute Error)** in hours — directly interpretable as scheduling buffer needed.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

reg_results = []

def eval_regressor(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    reg_results.append({'Model': name, 'MAE (hours)': round(mae,3),
                        'RMSE (hours)': round(rmse,3), 'R²': round(r2,3),
                        '_preds': y_pred})
    print(f'{name:35s}  MAE={mae:.3f}h  RMSE={rmse:.3f}h  R²={r2:.3f}')
    return model

imp = SimpleImputer(strategy='median')
X_train_imp = imp.fit_transform(X_train)
X_test_imp  = imp.transform(X_test)

# Baseline: mean predictor
dummy = DummyRegressor(strategy='mean')
eval_regressor('Mean Baseline', dummy, X_train_imp, y_train, X_test_imp, y_test)

# Random Forest
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=10,
                                min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
eval_regressor('Random Forest', rf_reg, X_train_imp, y_train, X_test_imp, y_test)

# Gradient Boosting
gb_reg = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                    max_depth=5, random_state=RANDOM_STATE)
eval_regressor('Gradient Boosting', gb_reg, X_train_imp, y_train, X_test_imp, y_test)

In [ ]:
# ── results table ─────────────────────────────────────────────────────────
reg_df = pd.DataFrame([{k:v for k,v in r.items() if k != '_preds'} for r in reg_results])
print('\n=== Duration Prediction — Model Comparison (§5.1) ===')
print(reg_df.to_string(index=False))
reg_df.to_csv(OUT_DIR / 'scheduling_regression_results.csv', index=False)

In [ ]:
# ── predicted vs actual plot ───────────────────────────────────────────────
best_reg = reg_results[-1]  # Gradient Boosting
y_pred_best = best_reg['_preds']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# scatter
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=8, color='#028090')
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Duration (hours)')
axes[0].set_ylabel('Predicted Duration (hours)')
axes[0].set_title(f'Gradient Boosting — Actual vs Predicted\n(MAE = {best_reg["MAE (hours)"]:.3f}h)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# residuals
residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=50, color='#00A896', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (hours)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'scheduling_regression_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── feature importance ─────────────────────────────────────────────────────
fi = pd.Series(gb_reg.feature_importances_, index=DURATION_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
fi.plot(kind='barh', ax=ax, color='#028090', edgecolor='white')
ax.set_title('Gradient Boosting — Feature Importances (job duration prediction)')
ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'scheduling_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4  Route Optimisation with OR-Tools

We simulate a single technician's working day:
- **8-hour working day**
- Sample N jobs from the dataset (with their lat/lng and ML-predicted durations)
- Compare three scheduling approaches:
  1. **Random order** (no optimisation)
  2. **Nearest-neighbour heuristic** (greedy — go to closest unvisited job)
  3. **OR-Tools VRP** (optimal routing given time constraints)

Metric: total travel distance (km), jobs completed within 8h.

In [ ]:
from math import radians, cos, sin, asin, sqrt

def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km."""
    R = 6371
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlng/2)**2
    return 2 * R * asin(sqrt(a))

def build_distance_matrix(locations):
    """NxN matrix of km distances. locations = list of (lat, lng)."""
    n = len(locations)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mat[i][j] = haversine_km(*locations[i], *locations[j])
    return mat

print('Distance utilities ready ✓')

In [ ]:
# ── sample a single day's jobs ─────────────────────────────────────────────
# Use a geographically compact sample (one Brazilian state) so travel times are realistic
state_sample = jobs[jobs['customer_state'] == 'SP'].dropna(subset=['lat','lng']).copy()

N_JOBS       = 15          # jobs available for the day
WORKING_HOURS = 8.0        # technician's working day
TRAVEL_SPEED_KMH = 30.0    # urban average

day_jobs = state_sample.sample(N_JOBS, random_state=RANDOM_STATE).reset_index(drop=True)

# predict durations with best model
day_X = imp.transform(day_jobs[DURATION_FEATURES].fillna(day_jobs[DURATION_FEATURES].median()))
day_jobs['predicted_duration_h'] = gb_reg.predict(day_X).clip(0.5, 8.0)

# depot: centroid of the day's jobs
depot_lat = day_jobs['lat'].mean()
depot_lng = day_jobs['lng'].mean()

# locations list: depot first, then jobs
locations = [(depot_lat, depot_lng)] + list(zip(day_jobs['lat'], day_jobs['lng']))
dist_matrix = build_distance_matrix(locations)

durations_h = [0.0] + list(day_jobs['predicted_duration_h'])  # depot has 0 duration

print(f'Sampled {N_JOBS} jobs in São Paulo state')
print(f'Total predicted work time: {sum(durations_h):.1f}h (working day: {WORKING_HOURS}h)')
print(f'Max completable without travel: {sum(1 for d in durations_h[1:] if d <= WORKING_HOURS)} jobs')

In [ ]:
# ── helper: simulate a route given a job visit order ──────────────────────
def simulate_route(order, dist_matrix, durations_h, working_hours, travel_speed):
    """
    order: list of job indices (1-based, depot=0)
    Returns: (jobs_completed, total_travel_km, total_travel_h, time_used_h)
    """
    current    = 0          # start at depot
    time_used  = 0.0
    travel_km  = 0.0
    completed  = 0

    for job_idx in order:
        dist   = dist_matrix[current][job_idx]
        travel = dist / travel_speed          # hours
        work   = durations_h[job_idx]

        if time_used + travel + work <= working_hours:
            time_used += travel + work
            travel_km += dist
            current    = job_idx
            completed += 1
        else:
            break  # can't fit the next job

    # return to depot
    travel_km += dist_matrix[current][0]
    return completed, round(travel_km, 2), round(time_used, 2)

job_indices = list(range(1, N_JOBS + 1))
print('Route simulator ready ✓')

In [ ]:
# ── Baseline 1: Random Order ───────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_STATE)
random_order = rng.permutation(job_indices).tolist()

rand_completed, rand_travel_km, rand_time = simulate_route(
    random_order, dist_matrix, durations_h, WORKING_HOURS, TRAVEL_SPEED_KMH)

print(f'Random order     : {rand_completed} jobs, {rand_travel_km:.1f} km travel')

In [ ]:
# ── Baseline 2: Nearest-Neighbour Heuristic ────────────────────────────────
def nearest_neighbour(depot_idx, job_indices, dist_matrix):
    current   = depot_idx
    unvisited = job_indices[:]
    route     = []
    while unvisited:
        nearest = min(unvisited, key=lambda j: dist_matrix[current][j])
        route.append(nearest)
        unvisited.remove(nearest)
        current = nearest
    return route

nn_order = nearest_neighbour(0, job_indices, dist_matrix)
nn_completed, nn_travel_km, nn_time = simulate_route(
    nn_order, dist_matrix, durations_h, WORKING_HOURS, TRAVEL_SPEED_KMH)

print(f'Nearest-neighbour: {nn_completed} jobs, {nn_travel_km:.1f} km travel')

In [ ]:
# ── Approach 3: OR-Tools VRP ───────────────────────────────────────────────
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def solve_vrp(dist_matrix, durations_h, working_hours, travel_speed):
    """
    Single-vehicle VRP with time windows (8h working day).
    Returns the ordered list of job indices in the optimal route.
    """
    n = len(dist_matrix)

    # Scale to integer minutes (OR-Tools works with integers)
    def travel_mins(i, j):
        return int(dist_matrix[i][j] / travel_speed * 60)

    def service_mins(i):
        return int(durations_h[i] * 60)

    manager = pywrapcp.RoutingIndexManager(n, 1, 0)  # n nodes, 1 vehicle, depot=0
    routing = pywrapcp.RoutingModel(manager)

    # ── transit callback ──────────────────────────────────────────────────
    def time_callback(from_idx, to_idx):
        from_node = manager.IndexToNode(from_idx)
        to_node   = manager.IndexToNode(to_idx)
        return travel_mins(from_node, to_node) + service_mins(from_node)

    transit_cb_idx = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_cb_idx)

    # ── time dimension (working day constraint) ───────────────────────────
    max_mins = int(working_hours * 60)
    routing.AddDimension(
        transit_cb_idx,
        0,            # no waiting time slack
        max_mins,     # maximum cumulative time
        True,         # cumulative from zero
        'Time'
    )

    # ── search parameters ─────────────────────────────────────────────────
    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    search_params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
    search_params.time_limit.seconds = 10

    solution = routing.SolveWithParameters(search_params)

    if not solution:
        print('OR-Tools: no solution found')
        return []

    # extract route order
    route = []
    index = routing.Start(0)
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        if node != 0:   # skip depot
            route.append(node)
        index = solution.Value(routing.NextVar(index))
    return route


ortools_order = solve_vrp(dist_matrix, durations_h, WORKING_HOURS, TRAVEL_SPEED_KMH)
or_completed, or_travel_km, or_time = simulate_route(
    ortools_order, dist_matrix, durations_h, WORKING_HOURS, TRAVEL_SPEED_KMH)

print(f'OR-Tools         : {or_completed} jobs, {or_travel_km:.1f} km travel')

## 5  Comparison Table (§5.1)

In [ ]:
sched_results = pd.DataFrame([
    {'Approach': 'Random Order (baseline)',   'Jobs Completed': rand_completed, 'Travel (km)': rand_travel_km},
    {'Approach': 'Nearest-Neighbour Heuristic','Jobs Completed': nn_completed,  'Travel (km)': nn_travel_km},
    {'Approach': 'OR-Tools Optimised',         'Jobs Completed': or_completed,  'Travel (km)': or_travel_km},
])

sched_results['Travel Reduction vs Random (%)'] = (
    (rand_travel_km - sched_results['Travel (km)']) / rand_travel_km * 100
).round(1)
sched_results['Extra Jobs vs Random'] = sched_results['Jobs Completed'] - rand_completed

print('\n=== Scheduling Optimisation — Approach Comparison (§5.1) ===')
print(sched_results.to_string(index=False))
sched_results.to_csv(OUT_DIR / 'scheduling_comparison.csv', index=False)

In [ ]:
# ── bar chart comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colours = ['#B0BEC5', '#028090', '#00A896']
labels  = ['Random\n(baseline)', 'Nearest\nNeighbour', 'OR-Tools\nOptimised']

# travel distance
travel_vals = [rand_travel_km, nn_travel_km, or_travel_km]
bars = axes[0].bar(labels, travel_vals, color=colours, edgecolor='white', width=0.5)
axes[0].set_ylabel('Total Travel Distance (km)')
axes[0].set_title('Travel Distance by Scheduling Approach')
for bar, val in zip(bars, travel_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f} km', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# jobs completed
jobs_vals = [rand_completed, nn_completed, or_completed]
bars2 = axes[1].bar(labels, jobs_vals, color=colours, edgecolor='white', width=0.5)
axes[1].set_ylabel('Jobs Completed (8h day)')
axes[1].set_title('Jobs Completed by Scheduling Approach')
axes[1].yaxis.set_major_locator(plt.MaxNLocator(integer=True))
for bar, val in zip(bars2, jobs_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'scheduling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/scheduling_comparison.png')

In [ ]:
# ── route map ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_route(ax, order, title, color, dist_matrix, locations):
    lats = [locations[i][0] for i in [0] + order + [0]]
    lngs = [locations[i][1] for i in [0] + order + [0]]
    # job points
    ax.scatter([l[1] for l in locations[1:]], [l[0] for l in locations[1:]],
               c='#455A64', s=50, zorder=3, label='Jobs')
    # depot
    ax.scatter(locations[0][1], locations[0][0],
               c='red', s=120, marker='*', zorder=4, label='Depot')
    # route line
    ax.plot(lngs, lats, color=color, linewidth=1.5, alpha=0.7)
    # number jobs in visit order
    for step, idx in enumerate(order):
        lat, lng = locations[idx]
        ax.annotate(str(step+1), (lng, lat), fontsize=7, ha='center', va='center',
                    color='white', fontweight='bold')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.grid(alpha=0.2)

plot_route(axes[0], random_order[:rand_completed], f'Random Order\n({rand_completed} jobs, {rand_travel_km:.0f} km)', '#B0BEC5', dist_matrix, locations)
plot_route(axes[1], nn_order[:nn_completed],       f'Nearest Neighbour\n({nn_completed} jobs, {nn_travel_km:.0f} km)',  '#028090', dist_matrix, locations)
plot_route(axes[2], ortools_order[:or_completed],  f'OR-Tools Optimised\n({or_completed} jobs, {or_travel_km:.0f} km)', '#00A896', dist_matrix, locations)

plt.tight_layout()
plt.savefig(OUT_DIR / 'scheduling_routes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/scheduling_routes.png')

## 6  RQ2 Answer Template (copy into §5.3)

In [ ]:
best_reg_row = reg_df.sort_values('MAE (hours)').iloc[0]
base_reg_row = reg_df.iloc[0]  # mean baseline

travel_reduction = (rand_travel_km - or_travel_km) / rand_travel_km * 100
extra_jobs = or_completed - rand_completed

print('=== RQ2 ANSWER TEMPLATE FOR §5.3 ===')
print(f"""
RQ2: Can ML-augmented scheduling optimisation meaningfully reduce travel time
and increase daily job capacity for micro-SME field service operators?

Yes. {best_reg_row['Model']} achieved a Mean Absolute Error of {best_reg_row['MAE (hours)']:.3f} hours
on job duration prediction, a {((base_reg_row['MAE (hours)'] - best_reg_row['MAE (hours)']) / base_reg_row['MAE (hours)'] * 100):.1f}% improvement over the mean-time baseline
(MAE = {base_reg_row['MAE (hours)']:.3f}h). When these ML-predicted durations were fed into the
Google OR-Tools VRP solver, the optimised schedule reduced total travel distance
by {travel_reduction:.1f}% compared to random ordering ({or_travel_km:.1f} km vs {rand_travel_km:.1f} km), and
completed {or_completed} jobs in an 8-hour working day versus {rand_completed} for the random baseline
(+{extra_jobs} additional jobs, a {extra_jobs/max(rand_completed,1)*100:.0f}% capacity increase).
""")

---
## Outputs Summary

| File | Used in |
|---|---|
| `outputs/scheduling_regression_results.csv` | Table in §5.1 |
| `outputs/scheduling_regression_plots.png` | Figure in §5.1 |
| `outputs/scheduling_feature_importance.png` | Figure in §5.1 |
| `outputs/scheduling_comparison.csv` | Table in §5.1 |
| `outputs/scheduling_comparison.png` | Figure in §5.1 |
| `outputs/scheduling_routes.png` | Figure in §5.1 |
| RQ2 template text | §5.3 |

**Next:** `03_heatmap.ipynb` — KDE spatial demand heat map from Olist customer lat/lng